# M3T2 - Consumidor y sinks con Spark Structured Streaming

Este cuaderno consume los eventos generados por `M3T2_productor_eventos_pedidos.ipynb`, aplica validaciones y escribe resultados en distintos sinks.

La práctica está pensada para trabajar conceptos de conectores, sinks, checkpoints, formatos de salida e idempotencia.

## 1. Preparación del entorno

Se preparan las carpetas de entrada, salida y checkpoint. Si ya existen, se eliminan para empezar la práctica desde cero.

In [ ]:
import os
import shutil

INPUT_DIR = os.path.abspath("input/pedidos_events")
RAW_OUTPUT_DIR = os.path.abspath("output/pedidos_raw_parquet")
AGG_OUTPUT_DIR = os.path.abspath("output/pedidos_metricas_parquet")
CSV_OUTPUT_DIR = os.path.abspath("output/pedidos_metricas_csv")
CHECKPOINT_RAW = os.path.abspath("checkpoint/pedidos_raw")
CHECKPOINT_AGG = os.path.abspath("checkpoint/pedidos_metricas")

for path in (RAW_OUTPUT_DIR, AGG_OUTPUT_DIR, CSV_OUTPUT_DIR, CHECKPOINT_RAW, CHECKPOINT_AGG):
    shutil.rmtree(path, ignore_errors=True)

os.makedirs(INPUT_DIR, exist_ok=True)

print("Entrada:", INPUT_DIR)
print("Salida raw:", RAW_OUTPUT_DIR)
print("Salida métricas:", AGG_OUTPUT_DIR)
print("Checkpoint raw:", CHECKPOINT_RAW)
print("Checkpoint métricas:", CHECKPOINT_AGG)

## 2. Instalación e inicialización de Spark

Si `pyspark` ya está instalado, puedes omitir la primera celda.

In [ ]:
# Descomenta si necesitas instalar PySpark en tu entorno
# %pip install pyspark==3.5.1

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("M3T2SinksStructuredStreaming")
    .master("local[*]")
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
spark

## 3. Definición del esquema

En streaming conviene declarar el esquema explícitamente para evitar inferencias inestables.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType

metadata_schema = StructType([
    StructField("schema_version", StringType(), True),
    StructField("producer", StringType(), True)
])

schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("source", StringType(), True),
    StructField("pedido_id", StringType(), True),
    StructField("cliente_id", StringType(), True),
    StructField("pais", StringType(), True),
    StructField("canal", StringType(), True),
    StructField("estado", StringType(), True),
    StructField("importe", DoubleType(), True),
    StructField("unidades", IntegerType(), True),
    StructField("metadata", metadata_schema, True)
])

schema

## 4. Lectura streaming desde ficheros JSON

El productor escribe JSON Lines en la carpeta de entrada. Spark lee los nuevos ficheros como microbatches.

In [ ]:
from pyspark.sql.functions import col, current_timestamp, window, count, sum as spark_sum, avg

eventos_df = (spark.readStream
    .schema(schema)
    .json(INPUT_DIR)
    .withColumn("processing_time", current_timestamp()))

eventos_validos_df = (eventos_df
    .filter(col("event_id").isNotNull())
    .filter(col("pedido_id").isNotNull())
    .filter(col("importe") >= 0))

eventos_validos_df.printSchema()

## 5. Sink raw en Parquet

Este sink conserva los eventos válidos originales. Es equivalente a una zona raw o bronze en un Data Lake.

In [ ]:
raw_query = (eventos_validos_df.writeStream
    .format("parquet")
    .option("path", RAW_OUTPUT_DIR)
    .option("checkpointLocation", CHECKPOINT_RAW)
    .outputMode("append")
    .start())

raw_query

## 6. Métricas agregadas por ventana

Se calculan métricas por país y ventanas de un minuto. La watermark permite cerrar ventanas aunque algunos eventos lleguen tarde.

In [ ]:
metricas_df = (eventos_validos_df
    .withWatermark("event_time", "2 minutes")
    .groupBy(window(col("event_time"), "1 minute"), col("pais"))
    .agg(
        count("*").alias("num_eventos"),
        spark_sum("importe").alias("importe_total"),
        avg("importe").alias("importe_medio"),
        spark_sum("unidades").alias("unidades_total")
    ))

metricas_df.printSchema()

## 7. Sink de métricas en memoria

El sink en memoria es útil para inspeccionar resultados durante el desarrollo.

In [ ]:
memory_query = (metricas_df.writeStream
    .format("memory")
    .queryName("metricas_pedidos")
    .outputMode("update")
    .start())

memory_query

Ejecuta esta consulta varias veces mientras el productor está generando eventos.

In [ ]:
spark.sql("""
SELECT pais, num_eventos, ROUND(importe_total, 2) AS importe_total, ROUND(importe_medio, 2) AS importe_medio
FROM metricas_pedidos
ORDER BY pais
""").show(truncate=False)

## 8. Ejemplo de salida compatible con Kafka

Para escribir en Kafka, el DataFrame debe tener una columna `value` y opcionalmente una columna `key`. La siguiente celda prepara esa estructura. Para ejecutar la escritura real se necesita un broker Kafka disponible.

In [ ]:
from pyspark.sql.functions import to_json, struct

kafka_ready_df = eventos_validos_df.select(
    col("pedido_id").cast("string").alias("key"),
    to_json(struct(
        "event_id", "event_type", "event_time", "source", "pedido_id",
        "cliente_id", "pais", "canal", "estado", "importe", "unidades"
    )).alias("value")
)

kafka_ready_df.printSchema()

In [ ]:
# Ejemplo opcional si tienes Kafka levantado:
# kafka_query = (kafka_ready_df.writeStream
#     .format("kafka")
#     .option("kafka.bootstrap.servers", "localhost:9092")
#     .option("topic", "pedidos_procesados")
#     .option("checkpointLocation", "checkpoint/pedidos_kafka")
#     .start())

## 9. Configuración de ejemplo para Kafka Connect Sink

Kafka Connect se configura normalmente con JSON. Este ejemplo muestra una plantilla conceptual para un sink de ficheros. En un entorno real se adaptaría el conector, rutas y credenciales.

In [ ]:
kafka_connect_sink_config = {
    "name": "pedidos-file-sink",
    "config": {
        "connector.class": "FileStreamSink",
        "tasks.max": "1",
        "topics": "pedidos_procesados",
        "file": "output/kafka_connect/pedidos_procesados.json",
        "key.converter": "org.apache.kafka.connect.storage.StringConverter",
        "value.converter": "org.apache.kafka.connect.json.JsonConverter",
        "value.converter.schemas.enable": "false"
    }
}

kafka_connect_sink_config

## 10. Parada ordenada de consultas

Detén las consultas cuando termines la práctica.

In [ ]:
for query in spark.streams.active:
    print("Deteniendo:", query.name, query.id)
    query.stop()

spark.streams.active